# Phase 3b: call against quarterly report

Plan: `plans/phase_3b.md`. Reports: `data/reports/` (UBS quarterly reports, JPMorgan Exhibit 99.2 supplements and, since the amendment of 2026-09-23, the UBS annual reports on Form 20-F for 2023 and 2024; all read only). Computation: `analysis/phase3b_reports.py` (text reconstruction and retrieval) and `analysis/phase3b_compare.py` (comparison table); outputs in `data/phase3b/`. Descriptive, no test.

The comparison table was compiled by reading the retrieved passages. Every quote is checked verbatim against its source by the script, and the column `checked_by_roman` is empty until reviewed.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase3b"
pd.set_option("display.max_colwidth", 160)
comp = pd.read_csv(OUT / "comparison.csv").fillna("")
cols = ["quarter", "in_call", "in_report", "figure_call", "figure_report", "first_or_more_precise", "comment"]
quotes = ["quarter", "quote_call", "location_call", "quote_report", "location_report"]

## Text reconstruction

In [ ]:
pd.read_csv(OUT / "checks.csv")

Every file yields text on every page except two pages in each annual report, which are blank or carry only an image. For all four check words the reconstructed text contains at least as many occurrences as the raw file; `FRTB` does not occur as a whole word in the raw UBS files before 2024-Q4 and is found only there after reconstruction too, so the earlier reports use the long form “Fundamental Review of the Trading Book”.

In [ ]:
hr, hc = pd.read_csv(OUT / "hits_reports.csv"), pd.read_csv(OUT / "hits_calls.csv")
pd.concat({"report hits": hr.groupby(["case", "quarter"]).size().unstack("case"),
           "call hits": hc.groupby(["case", "quarter"]).size().unstack("case")}, axis=1).fillna(0).astype(int)

The retrieval patterns with a window of non-full-stop characters miss sentences that contain a decimal number (“issued 3.5 billion in AT1”); such sentences were found by reading the phase 3 statements around the hits and are cited with their `stmt_id`.

In [ ]:
pd.crosstab(comp.case, comp.first_or_more_precise).rename(index={
    "C1": "C1 UBS gross cost saves", "C2": "C2 UBS integration-related expenses", "C3": "C3 UBS Basel III final / FRTB",
    "C4": "C4 UBS AT1 issuance", "C5": "C5 JPM First Republic"})

## C1: UBS gross cost saves

In [ ]:
comp[comp.case == "C1"][cols]

Both documents carry the target and its raise from over 10 to 13 billion on the same day (2023-Q4). The call reports progress earlier (around 3 billion in 2023-Q3, when the report gives only the target), while in 2024-Q2 and 2024-Q3 the report is more precise, with the cumulative level, the share of the target and the raised year-end expectation.

## C2: UBS integration-related expenses

In [ ]:
comp[comp.case == "C2"][cols]

The explicit expected total of integration-related expenses (around 13 billion from 2023-Q4, raised to around 14 billion in 2024-Q4) and the cumulative amounts to date appear only in the calls, including against the annual reports on Form 20-F. The reports give quarterly amounts, next-quarter expectations and, in 2023-Q2, an indirect order of magnitude (expenses broadly offset by accretion-to-par effects of approximately USD 12bn, with a revised estimate announced for 4Q23). A reader of the reports alone therefore sees the scale but not the explicit total or its increase by about 1 billion.

## Annual reports on Form 20-F (amendment of 2026-09-23)

In [ ]:
comp[comp.quarter.str.contains("annual")][["case"] + cols]

Both annual reports state the savings target of around 13 billion with its progress (around 4 billion at the end of 2023, 7.5 billion or 58 % at the end of 2024), matching the fourth-quarter calls; the 2023 report adds a milestone of around 45 % by the end of 2024. Neither annual report states the expected total of integration-related expenses or its increase from around 13 to around 14 billion: the only 13 billion figures there refer to the savings target.

## C3: UBS Basel III final and FRTB

In [ ]:
comp[comp.case == "C3"][cols]

The call gives the first figure (a roughly 5 % RWA increase, 2023-Q3), but from 2023-Q4 the report is more complete: 25 billion for the Group including 10 billion in Non-core and Legacy, against 15 billion for the core businesses in the call. The downward revision to a low single-digit percentage (2024-Q3) and to 1 billion (2024-Q4) is in both documents on the same day; the capital-ratio effects and the finer split of the FRTB effect (market risk and CVA) are only in the report.

## C4: UBS AT1 issuance

In [ ]:
comp[comp.case == "C4"][cols]

The plan (up to 2 billion in 2024) and the annual total (3.5 billion, with an ambition of 4.3 % of RWA) are stated only in the calls, while the individual issuances per quarter (1.5, 0.4 and 1.6 billion) are in the reports, one of them (0.4 billion in 2024-Q2) only there. The phase 3 rules found no follow-up to the plan; the reports confirm that it was exceeded.

## C5: JPMorgan and First Republic

In [ ]:
comp[comp.case == "C5"][cols]

The event-call statements on restructuring costs (about 2 billion post-tax) and incremental net income (more than 500 million per year) are not restated as numbers in any later call or supplement. The supplements carry a separate First Republic page from 2023-Q2 to 2024-Q1 and show the revision of the bargain purchase gain from 2.7 to 2.8 billion, which the calls never mention; after 2024-Q1 separate disclosure ends in both documents, and the supplements never contain an outlook.

## Quotes and locations

In [ ]:
comp[quotes + ["case"]].set_index(["case", "quarter"])

## Reading

For the UBS merger topics the call is not redundant: the explicit expected total integration cost and its increase (the reports give only an indirect order of magnitude in 2023-Q2), the AT1 plan and the first Basel III final estimate appear only in the calls. For the regulatory capital effects the reports are the more complete source from 2023-Q4 onwards, and for JPMorgan the supplements are more precise on First Republic but neither document tracks the merger-specific targets of the event call.